# Lab 2 · Clean, connect and check

**In class:** Part A, guided, on a messy version of the penguins data · **At home:** Part B, the same steps on your own two tables
**Due:** Sunday, November 8 at 11:59 PM (extended for Election Day) · **Submit:** your repo URL on CourseWorks

Tonight's three skills:
1. **Clean** a messy CSV with pandas and load it into DuckDB
2. **Connect** two tables with a join, on a key you've checked
3. **Check** every join by counting rows before and after

Run cells top to bottom with **Shift+Enter**.

---
# Part A · Guided: messy penguins + a lookup table

## A1. Read the messy file with pandas

`penguins_raw.csv` is the same study as last week's penguins, in its original form: long column names, extra fields, comments. **pandas** is a Python library for working with tables; a table in pandas is called a **DataFrame**.

In [45]:
from pathlib import Path
import pandas as pd
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / "data" / "penguins_raw.csv")

print(raw.shape)   # (rows, columns)
raw.head()

(344, 17)


,studyName,Sample Number,Species,Region,Island,Stage,Individual ID,Clutch Completion,Date Egg,Culmen Length (mm),Culmen Depth (mm),Flipper Length (mm),Body Mass (g),Sex,Delta 15 N (o/oo),Delta 13 C (o/oo),Comments
0,PAL0708,1,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A1,Yes,2007-11-11,39.1,18.7,181.0,3750.0,MALE,NaN,NaN,Not enough blood for isotopes.
1,PAL0708,2,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N1A2,Yes,2007-11-11,39.5,17.4,186.0,3800.0,FEMALE,8.94956,-24.69454,NaN
2,PAL0708,3,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A1,Yes,2007-11-16,40.3,18.0,195.0,3250.0,FEMALE,8.36821,-25.33302,NaN
3,PAL0708,4,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N2A2,Yes,2007-11-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adult not sampled.
4,PAL0708,5,Adelie Penguin (Pygoscelis adeliae),Anvers,Torgersen,"Adult, 1 Egg Stage",N3A1,Yes,2007-11-16,36.7,19.3,193.0,3450.0,FEMALE,8.76651,-25.32426,NaN


How much is missing in each column? `isna()` marks missing values; `.sum()` counts them per column.

In [46]:
raw.isna().sum()

studyName                0
Sample Number            0
Species                  0
Region                   0
Island                   0
Stage                    0
Individual ID            0
Clutch Completion        0
Date Egg                 0
Culmen Length (mm)       2
Culmen Depth (mm)        2
Flipper Length (mm)      2
Body Mass (g)            2
Sex                     11
Delta 15 N (o/oo)       14
Delta 13 C (o/oo)       13
Comments               290
dtype: int64

Look at the species column. What would make it hard to use as a key?

In [47]:
raw["Species"].value_counts()

Species
Adelie Penguin (Pygoscelis adeliae)          152
Gentoo penguin (Pygoscelis papua)            124
Chinstrap penguin (Pygoscelis antarctica)     68
Name: count, dtype: int64

**Notice:** long names with the Latin name in parentheses, and inconsistent capitals ("Adelie **P**enguin" vs. "Gentoo **p**enguin"). Text that *looks* the same to a person often won't match exactly in a join.

## A2. Three cleaning steps

**Step 1. Rename columns** to short, lowercase names with no spaces, so they're easy to type in SQL.

In [48]:
penguins = raw.rename(columns={
    "studyName": "study",
    "Individual ID": "individual_id",
    "Species": "species_raw",
    "Island": "island",
    "Date Egg": "date_egg",
    "Clutch Completion": "clutch_completion",
    "Culmen Length (mm)": "bill_length_mm",
    "Culmen Depth (mm)": "bill_depth_mm",
    "Flipper Length (mm)": "flipper_length_mm",
    "Body Mass (g)": "body_mass_g",
    "Sex": "sex",
})
penguins.columns.tolist()

['study',
 'Sample Number',
 'species_raw',
 'Region',
 'island',
 'Stage',
 'individual_id',
 'clutch_completion',
 'date_egg',
 'bill_length_mm',
 'bill_depth_mm',
 'flipper_length_mm',
 'body_mass_g',
 'sex',
 'Delta 15 N (o/oo)',
 'Delta 13 C (o/oo)',
 'Comments']

**Step 2. Standardize text.** Make `sex` lowercase, and pull the Latin name out of the parentheses into its own column. Scientific names are written the same way everywhere, which makes them a better key than the common name.

In [49]:
penguins["sex"] = penguins["sex"].str.lower()
penguins["latin_name"] = penguins["species_raw"].str.extract(r"\((.*)\)")   # text inside ( )

penguins[["species_raw", "latin_name", "sex"]].drop_duplicates().head(6)

,species_raw,latin_name,sex
0,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,male
1,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,female
3,Adelie Penguin (Pygoscelis adeliae),Pygoscelis adeliae,NaN
152,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,female
153,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,male
178,Gentoo penguin (Pygoscelis papua),Pygoscelis papua,NaN


**Step 3. Fix types and keep what you need.** The egg date came in as text; make it a real date. Then keep only the columns we'll use.

In [50]:
penguins["date_egg"] = pd.to_datetime(penguins["date_egg"])

penguins = penguins[[
    "study", "individual_id", "latin_name", "island", "date_egg", "clutch_completion",
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex",
]]
penguins.dtypes

study                           str
individual_id                   str
latin_name                      str
island                          str
date_egg             datetime64[us]
clutch_completion               str
bill_length_mm              float64
bill_depth_mm               float64
flipper_length_mm           float64
body_mass_g                 float64
sex                             str
dtype: object

## A3. Load it into DuckDB and verify with SQL

DuckDB can read a pandas DataFrame directly by its variable name. Always check the row count after moving data: **rows in should equal rows out.**

In [51]:
con = duckdb.connect(str(ROOT / "lab2.duckdb"))
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

print("rows in pandas:", len(penguins))
con.sql("SELECT count(*) AS rows_in_duckdb FROM penguins_clean")

rows in pandas: 344


┌────────────────┐
│ rows_in_duckdb │
│     int64      │
├────────────────┤
│            344 │
└────────────────┘

In [52]:
con.sql("DESCRIBE penguins_clean")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ study             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ individual_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latin_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date_egg          │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ clutch_completion │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_

## A4. What is one row? Is the ID really an ID?

There's a column called `individual_id`. If it identified one penguin, every penguin would appear once per season. Let's check.

In [53]:
con.sql("""
    SELECT count(*)                      AS rows,
           count(DISTINCT individual_id) AS distinct_ids
    FROM penguins_clean
""")

┌───────┬──────────────┐
│ rows  │ distinct_ids │
│ int64 │    int64     │
├───────┼──────────────┤
│   344 │          190 │
└───────┴──────────────┘

In [54]:
con.sql("""
    SELECT study, latin_name, island, individual_id
    FROM penguins_clean
    WHERE individual_id = 'N11A1'
""")

┌─────────┬────────────────────┬─────────┬───────────────┐
│  study  │     latin_name     │ island  │ individual_id │
│ varchar │      varchar       │ varchar │    varchar    │
├─────────┼────────────────────┼─────────┼───────────────┤
│ PAL0708 │ Pygoscelis adeliae │ Biscoe  │ N11A1         │
│ PAL0809 │ Pygoscelis papua   │ Biscoe  │ N11A1         │
└─────────┴────────────────────┴─────────┴───────────────┘

**The same ID belongs to an Adélie penguin in one season and a Gentoo the next.** It can't be one bird. It's most likely a nest-and-adult label the researchers reused. So:

- `individual_id` **alone is not a key.** A column named "ID" isn't automatically unique; always check with `count(DISTINCT ...)`.
- **One row = one adult penguin at one nest in one breeding season.** That's the table's **grain**, and you should be able to say it for every table you use.

## A5. A lookup table to join

`species_lookup.csv` is a small reference table: one row per species, with its common name, Latin name, genus and group.

In [55]:
con.sql(f"CREATE OR REPLACE TABLE species AS SELECT * FROM read_csv('{ROOT / 'data' / 'species_lookup.csv'}')")
con.sql("SELECT * FROM species")

┌───────────────────┬────────────────────────┬─────────────┬───────────────────────┐
│    common_name    │       latin_name       │    genus    │      group_name       │
│      varchar      │        varchar         │   varchar   │        varchar        │
├───────────────────┼────────────────────────┼─────────────┼───────────────────────┤
│ Adélie penguin    │ Pygoscelis adeliae     │ Pygoscelis  │ brush-tailed penguins │
│ Chinstrap penguin │ Pygoscelis antarcticus │ Pygoscelis  │ brush-tailed penguins │
│ Gentoo penguin    │ Pygoscelis papua       │ Pygoscelis  │ brush-tailed penguins │
│ Emperor penguin   │ Aptenodytes forsteri   │ Aptenodytes │ great penguins        │
└───────────────────┴────────────────────────┴─────────────┴───────────────────────┘

Before joining, check that the key is **unique** in the lookup table. If a key appeared twice, every matching penguin would be copied twice.

In [56]:
con.sql("""
    SELECT latin_name, count(*) AS n
    FROM species
    GROUP BY latin_name
    HAVING count(*) > 1
""")   # no rows back = the key is unique

┌────────────┬───────┐
│ latin_name │   n   │
│  varchar   │ int64 │
└────────────┴───────┘
        0 rows      

## A6. Join, and count what comes out

`JOIN ... ON` lines up rows where the key matches. We expect **344** rows out, one per penguin row.

In [57]:
con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             276 │
└─────────────────┘

**Only 276, not 344.** An inner `JOIN` silently drops rows that don't match. Nothing errored. This is a data cascade in the making.

## A7. Find what didn't match with a LEFT JOIN

A `LEFT JOIN` keeps every penguin row. Where there's no match, the lookup's columns come back empty (`NULL`), so we can find the problem rows.

In [58]:
con.sql("""
    SELECT p.latin_name, count(*) AS unmatched_rows
    FROM penguins_clean p
    LEFT JOIN species s ON p.latin_name = s.latin_name
    WHERE s.latin_name IS NULL
    GROUP BY p.latin_name
""")

┌───────────────────────┬────────────────┐
│      latin_name       │ unmatched_rows │
│        varchar        │     int64      │
├───────────────────────┼────────────────┤
│ Pygoscelis antarctica │             68 │
└───────────────────────┴────────────────┘

Compare with the lookup: the data says **Pygoscelis antarctic*a***, the lookup says **Pygoscelis antarctic*us***. Two sources spelled the same species differently, and 68 penguins vanished from the join.

## A8. Fix the key, then join again

Fix it at the source of the problem (the cleaning step), then reload and rejoin.

In [59]:
penguins["latin_name"] = penguins["latin_name"].replace(
    {"Pygoscelis antarctica": "Pygoscelis antarcticus"}
)
con.sql("CREATE OR REPLACE TABLE penguins_clean AS SELECT * FROM penguins")

con.sql("""
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
""")   # now 344

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             344 │
└─────────────────┘

Which way you join matters. Start from the **lookup** with a `LEFT JOIN` to see species that have no penguins:

In [60]:
con.sql("""
    SELECT s.common_name, count(p.latin_name) AS penguin_rows
    FROM species s
    LEFT JOIN penguins_clean p ON p.latin_name = s.latin_name
    GROUP BY s.common_name
    ORDER BY penguin_rows DESC
""")

┌───────────────────┬──────────────┐
│    common_name    │ penguin_rows │
│      varchar      │    int64     │
├───────────────────┼──────────────┤
│ Adélie penguin    │          152 │
│ Gentoo penguin    │          124 │
│ Chinstrap penguin │           68 │
│ Emperor penguin   │            0 │
└───────────────────┴──────────────┘

## A9. What a duplicate key does

Here we fake a lookup table that lists Gentoo twice, then join. Watch the row count.

In [61]:
con.sql("""
    WITH species_with_dupe AS (
        SELECT * FROM species
        UNION ALL
        SELECT * FROM species WHERE common_name = 'Gentoo penguin'
    )
    SELECT count(*) AS rows_after_join
    FROM penguins_clean p
    JOIN species_with_dupe s ON p.latin_name = s.latin_name
""")   # 468: every Gentoo row was copied

┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│             468 │
└─────────────────┘

**344 became 468**, and every total or average involving Gentoos would now be wrong. That's why you check key uniqueness (A5) before every join.

## A10. Use the join to answer a question

Now that the tables connect, ask a question that needs both: the lookup's readable names, the penguin table's measurements.

In [62]:
con.sql("""
    SELECT s.common_name,
           p.island,
           count(*)                    AS penguins,
           round(avg(p.body_mass_g))   AS avg_mass_g
    FROM penguins_clean p
    JOIN species s ON p.latin_name = s.latin_name
    GROUP BY s.common_name, p.island
    ORDER BY s.common_name, p.island
""")

┌───────────────────┬───────────┬──────────┬────────────┐
│    common_name    │  island   │ penguins │ avg_mass_g │
│      varchar      │  varchar  │  int64   │   double   │
├───────────────────┼───────────┼──────────┼────────────┤
│ Adélie penguin    │ Biscoe    │       44 │     3710.0 │
│ Adélie penguin    │ Dream     │       56 │     3688.0 │
│ Adélie penguin    │ Torgersen │       52 │     3706.0 │
│ Chinstrap penguin │ Dream     │       68 │     3733.0 │
│ Gentoo penguin    │ Biscoe    │      124 │     5076.0 │
└───────────────────┴───────────┴──────────┴────────────┘

### The join checklist (use it in Part B)

1. **Grain:** what is one row in each table?
2. **Key:** which column connects them, and is it **unique** in the lookup side?
3. **Clean the key** the same way on both sides (spelling, capitals, spaces, codes vs. names, types).
4. **Count rows** before and after. More rows = a duplicate key. Fewer = unmatched keys.
5. **LEFT JOIN … IS NULL** to see exactly what didn't match, then decide: fix, drop, or keep as unknown.

---
# Part B · At home: your own two tables

Apply the checklist to the two tables you chose in Lab 1. If you'd like to swap datasets, that's fine; you just need two public tables that share a column.

## B1. Load both tables with pandas

In [71]:
FILE_A = "Queens_HPD_Complaints_Problems_2025.csv"     # <- your main table's file in data/
FILE_B = "Queens_HPD_Violations_2025.csv"   # <- the related table's file in data/

a = pd.read_csv(ROOT / "data" / FILE_A)
b = pd.read_csv(ROOT / "data" / FILE_B)
print("A:", a.shape, "  B:", b.shape)
a.head()

A: (107497, 33)   B: (106400, 41)


,received_date,problem_id,complaint_id,building_id,borough,house_number,street_name,post_code,block,lot,...,problem_duplicate_flag,complaint_anonymous_flag,unique_key,latitude,longitude,council_district,census_tract,bin,bbl,nta
0,2025-01-01T00:34:41.000,26499011,13474352,608285,QUEENS,92-38,220 STREET,11428.0,10705,24,...,N,N,63586555.0,40.724304,-73.737592,23.0,560.0,4229355.0,4.107050e+09,Queens Village
1,2025-01-01T00:34:59.000,26499012,13474353,432844,QUEENS,31-63,44 STREET,11103.0,709,16,...,Y,N,63595102.0,40.758456,-73.914997,26.0,153.0,4012485.0,4.007090e+09,Astoria (East)-Woodside (North)
2,2025-01-01T00:37:45.000,26499013,13474354,484616,QUEENS,37-06,81 STREET,11372.0,1291,6,...,Y,Y,63587730.0,40.749329,-73.885194,25.0,283.0,4029987.0,4.012910e+09,Jackson Heights
3,2025-01-01T00:40:59.000,26499014,13474355,432844,QUEENS,31-63,44 STREET,11103.0,709,16,...,Y,N,63591400.0,40.758456,-73.914997,26.0,153.0,4012485.0,4.007090e+09,Astoria (East)-Woodside (North)
4,2025-01-01T00:52:14.000,26499018,13474359,884620,QUEENS,90-23,175 STREET,11432.0,9811,35,...,N,N,63587683.0,40.709595,-73.786532,27.0,462.0,4532685.0,4.098110e+09,Jamaica


In [72]:
b.head()

,violationid,buildingid,registrationid,boroid,boro,housenumber,lowhousenumber,highhousenumber,streetname,streetcode,...,violationstatus,rentimpairing,latitude,longitude,communityboard,councildistrict,censustract,bin,bbl,nta
0,135101,415601,414257,4,QUEENS,33-08,33-08,33-08,29 STREET,7740,...,Close,N,40.761756,-73.927782,1.0,22.0,53.0,4007207.0,4.005820e+09,Astoria (Central)
1,135102,415601,414257,4,QUEENS,33-08,33-08,33-08,29 STREET,7740,...,Close,N,40.761756,-73.927782,1.0,22.0,53.0,4007207.0,4.005820e+09,Astoria (Central)
2,135103,415601,414257,4,QUEENS,33-08,33-08,33-08,29 STREET,7740,...,Close,N,40.761756,-73.927782,1.0,22.0,53.0,4007207.0,4.005820e+09,Astoria (Central)
3,139230,417114,405110,4,QUEENS,23-12,23-12,23-12,31 AVENUE,7990,...,Close,N,40.766554,-73.928228,1.0,22.0,75.0,4006650.0,4.005690e+09,Astoria (Central)
4,149056,421846,413250,4,QUEENS,34-38,34-38,34-38,34 STREET,8540,...,Close,N,40.757903,-73.925390,1.0,26.0,57.0,4008249.0,4.006100e+09,Astoria (Central)


**Grain** (double-click to edit). One row in table A is: ___. One row in table B is: ___.

## B2. Clean: at least three steps

Use the Part A steps as a model: rename columns, standardize text, fix types, drop columns you don't need. Write each step in its own line with a comment saying *why*.

In [73]:
# Step 1: rename columns (short, lowercase, no spaces)
# a = a.rename(columns={"Country Code": "country_code", ...})

# Step 2: standardize the key text so both tables match
# a["country_code"] = a["country_code"].str.strip().str.upper()
# b["country_code"] = b["country_code"].str.strip().str.upper()

# Step 3: fix a type, or drop rows/columns you don't need
# a["year"] = a["year"].astype(int)

a.head()

,received_date,problem_id,complaint_id,building_id,borough,house_number,street_name,post_code,block,lot,...,problem_duplicate_flag,complaint_anonymous_flag,unique_key,latitude,longitude,council_district,census_tract,bin,bbl,nta
0,2025-01-01T00:34:41.000,26499011,13474352,608285,QUEENS,92-38,220 STREET,11428.0,10705,24,...,N,N,63586555.0,40.724304,-73.737592,23.0,560.0,4229355.0,4.107050e+09,Queens Village
1,2025-01-01T00:34:59.000,26499012,13474353,432844,QUEENS,31-63,44 STREET,11103.0,709,16,...,Y,N,63595102.0,40.758456,-73.914997,26.0,153.0,4012485.0,4.007090e+09,Astoria (East)-Woodside (North)
2,2025-01-01T00:37:45.000,26499013,13474354,484616,QUEENS,37-06,81 STREET,11372.0,1291,6,...,Y,Y,63587730.0,40.749329,-73.885194,25.0,283.0,4029987.0,4.012910e+09,Jackson Heights
3,2025-01-01T00:40:59.000,26499014,13474355,432844,QUEENS,31-63,44 STREET,11103.0,709,16,...,Y,N,63591400.0,40.758456,-73.914997,26.0,153.0,4012485.0,4.007090e+09,Astoria (East)-Woodside (North)
4,2025-01-01T00:52:14.000,26499018,13474359,884620,QUEENS,90-23,175 STREET,11432.0,9811,35,...,N,N,63587683.0,40.709595,-73.786532,27.0,462.0,4532685.0,4.098110e+09,Jamaica


In [83]:
import pandas as pd

# Step 1: Rename the building ID column so both tables use the same name
b = b.rename(columns={"buildingid": "building_id"})

# Step 2: Standardize borough names for consistent comparison
a["borough"] = a["borough"].str.strip().str.upper()
b["boro"] = b["boro"].str.strip().str.upper()

# Step 3: Convert building IDs to the same numeric type for joining
a["building_id"] = pd.to_numeric(a["building_id"], errors="coerce").astype("Int64")
b["building_id"] = pd.to_numeric(b["building_id"], errors="coerce").astype("Int64")

a.head()

,received_date,problem_id,complaint_id,building_id,borough,house_number,street_name,post_code,block,lot,...,problem_duplicate_flag,complaint_anonymous_flag,unique_key,latitude,longitude,council_district,census_tract,bin,bbl,nta
0,2025-01-01T00:34:41.000,26499011,13474352,608285,QUEENS,92-38,220 STREET,11428.0,10705,24,...,N,N,63586555.0,40.724304,-73.737592,23.0,560.0,4229355.0,4.107050e+09,Queens Village
1,2025-01-01T00:34:59.000,26499012,13474353,432844,QUEENS,31-63,44 STREET,11103.0,709,16,...,Y,N,63595102.0,40.758456,-73.914997,26.0,153.0,4012485.0,4.007090e+09,Astoria (East)-Woodside (North)
2,2025-01-01T00:37:45.000,26499013,13474354,484616,QUEENS,37-06,81 STREET,11372.0,1291,6,...,Y,Y,63587730.0,40.749329,-73.885194,25.0,283.0,4029987.0,4.012910e+09,Jackson Heights
3,2025-01-01T00:40:59.000,26499014,13474355,432844,QUEENS,31-63,44 STREET,11103.0,709,16,...,Y,N,63591400.0,40.758456,-73.914997,26.0,153.0,4012485.0,4.007090e+09,Astoria (East)-Woodside (North)
4,2025-01-01T00:52:14.000,26499018,13474359,884620,QUEENS,90-23,175 STREET,11432.0,9811,35,...,N,N,63587683.0,40.709595,-73.786532,27.0,462.0,4532685.0,4.098110e+09,Jamaica


## B3. Load into DuckDB and check the key

In [86]:
con = duckdb.connect("lab2.duckdb")
con.sql("CREATE OR REPLACE TABLE table_a AS SELECT * FROM a")
con.sql("CREATE OR REPLACE TABLE table_b AS SELECT * FROM b")

KEY_A = "building_id"   # <- after your renames
KEY_B = "building_id"

# Is the key unique in table B? No rows back = unique.
con.sql(f"""
    SELECT "{KEY_B}", COUNT(*) AS n
    FROM table_b
    GROUP BY "{KEY_B}"
    HAVING COUNT(*) > 1
    ORDER BY n DESC
    LIMIT 10
""")

┌─────────────┬───────┐
│ building_id │   n   │
│    int64    │ int64 │
├─────────────┼───────┤
│      687226 │   944 │
│      424613 │   512 │
│      694083 │   470 │
│      421489 │   436 │
│      484619 │   395 │
│      415206 │   347 │
│      489176 │   342 │
│      812275 │   312 │
│      642725 │   307 │
│      811258 │   306 │
└─────────────┴───────┘
  10 rows   2 columns

## B4. Join and count

In [87]:
print("rows in table A:", con.sql("SELECT count(*) FROM table_a").fetchone()[0])

con.sql(f"""
    SELECT count(*) AS rows_after_join
    FROM table_a a
    JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
""")

rows in table A: 107497


┌─────────────────┐
│ rows_after_join │
│      int64      │
├─────────────────┤
│         5102631 │
└─────────────────┘

## B5. Find what didn't match

In [88]:
con.sql(f"""
    SELECT a."{KEY_A}", count(*) AS unmatched_rows
    FROM table_a a
    LEFT JOIN table_b b ON a."{KEY_A}" = b."{KEY_B}"
    WHERE b."{KEY_B}" IS NULL
    GROUP BY a."{KEY_A}"
    ORDER BY unmatched_rows DESC
    LIMIT 20
""")

┌─────────────┬────────────────┐
│ building_id │ unmatched_rows │
│    int64    │     int64      │
├─────────────┼────────────────┤
│      491053 │            332 │
│      661708 │            153 │
│      660867 │            114 │
│      643177 │             70 │
│      657932 │             69 │
│     1001572 │             56 │
│      498419 │             55 │
│      424623 │             48 │
│      486562 │             43 │
│      910263 │             41 │
│      504327 │             39 │
│      811229 │             38 │
│      687263 │             37 │
│      505175 │             32 │
│      426670 │             32 │
│      493556 │             30 │
│      417673 │             29 │
│      406825 │             28 │
│      810887 │             28 │
│      679726 │             27 │
└─────────────┴────────────────┘
  20 rows            2 columns

**What didn't match, and what will you do about it?** (fix the key, drop those rows, or keep them as unknown, and why)



## B6. Answer one question that needs both tables

Write a query that joins the tables and uses columns from **both**, e.g. average GDP by region, complaints per 1,000 residents by ZIP code.

In [89]:
con.sql("""
    -- your query here: join table_a and table_b, then GROUP BY a column from table_b
    SELECT * FROM table_a LIMIT 5
""")

┌─────────────────────────┬────────────┬──────────────┬─────────────┬─────────┬──────────────┬─────────────┬───────────┬───────┬───────┬───────────┬─────────────────┬───────────────┬──────────────────┬───────────┬────────────────┬─────────────────┬──────────────────────────┬──────────────────┬─────────────────────────┬────────────────┬─────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────┬──────────────────────────┬────────────┬───────────┬────────────┬──────────────────┬──────────────┬───────────┬──────────────┬─────────────────────────────────┐
│      received_date      │ problem_id │ complaint_id │ building_id │ borough │ house_n

**What does the result tell you?** (one or two sentences)

## B7. Submit

1. Run the cell below.
2. Answer the two reflection questions in `labs/lab2.md`.
3. Save this notebook (Cmd+S), then commit and push: **Source Control** → `Lab 2` → **Commit** → **Sync Changes**.
4. Paste your repo URL into the Lab 2 assignment on CourseWorks.

In [90]:
con.close()
print("Done. Save, commit, push, and submit your repo URL.")

Done. Save, commit, push, and submit your repo URL.
